# Camada Silver — ecommerce_pedidos

## Objetivo

Aplicar as regras de Data Quality sobre os dados provenientes da camada Bronze
da tabela `ecommerce_pedidos`.

O processamento deve permitir:

- identificar registros válidos;
- identificar registros rejeitados;
- registrar os motivos de rejeição;
- preparar os dados certificados para persistência na camada Silver.

Fluxo:

Bronze → Regras de Data Quality → Válidos / Rejeitados → Silver

In [0]:
# Leitura da camada Bronze

df_bronze = spark.table(
    "workspace.bronze.ecommerce_pedidos"
)

print("Registros recebidos da Bronze:", df_bronze.count())

df_bronze.printSchema()

In [0]:
# Verifica as cargas existentes na Bronze para entender
# como os registros estão distribuídos entre as execuções.

from pyspark.sql import functions as F

df_bronze.groupBy(
    "data_carga"
).count().orderBy(
    F.col("data_carga").desc()
).show(truncate=False)

In [0]:
# Define os valores permitidos para campos com domínio fechado.

status_validos = [
    "Processando",
    "Pagamento Aprovado",
    "Em Separação",
    "Enviado",
    "Entregue",
    "Cancelado"
]

metodos_pagamento_validos = [
    "cartão crédito",
    "cartão débito",
    "pix",
    "boleto"
]

In [0]:
# Aplica as primeiras regras de Data Quality e cria indicadores de falha por registro.

from pyspark.sql import functions as F
from pyspark.sql.window import Window

janela_id_pedido = Window.partitionBy("id_pedido")

df_dq = (
    df_bronze

      # Coluna auxiliar: conta quantas vezes cada id_pedido aparece.
    .withColumn(
        "qtd_id_pedido",
        F.count("*").over(janela_id_pedido)
    )

    # Diagnóstico: identifica id_pedido nulo.
    .withColumn(
        "dq_id_pedido_nulo",
        F.col("id_pedido").isNull()
    )

    # Diagnóstico: identifica id_pedido repetido.
    .withColumn(
        "dq_id_pedido_duplicado",
        F.col("id_pedido").isNotNull()
        & (F.col("qtd_id_pedido") > 1)
    )

    # Resultado final da Regra 1:
    # falha se o id_pedido for nulo OU duplicado.
    .withColumn(
        "dq_regra_01_id_pedido",
        F.col("dq_id_pedido_nulo")
        | F.col("dq_id_pedido_duplicado")
    )
    # 2. id_cliente não pode ser nulo;
    .withColumn(
        "dq_id_cliente_nulo",
        F.col("id_cliente").isNull()
    )

    # 3. status_pedido deve estar no domínio permitido;
    .withColumn(
        "dq_status_invalido",
        F.col("status_pedido").isNull()
        | ~F.col("status_pedido").isin(status_validos)
    )

    # 4. valor_total deve ser maior que zero;
    .withColumn(
        "dq_valor_total_invalido",
        F.col("valor_total").isNull()
        | (F.col("valor_total") <= 0)
    )

    # 5. metodo_pagamento deve estar no domínio permitido.
    .withColumn(
        "dq_metodo_pagamento_invalido",
        F.col("metodo_pagamento").isNull()
        | ~F.col("metodo_pagamento").isin(
            metodos_pagamento_validos
        )
    )

    # Regra 6 - A última atualização de status não pode ser anterior ao pedido.
    # True = falhou na regra | False = passou na regra.
    .withColumn(
    "dq_regra_06_data_status",
    F.col("dt_pedido").isNotNull()
    & F.col("dt_ultima_atualizacao_status").isNotNull()
    & (
        F.col("dt_ultima_atualizacao_status")
        < F.col("dt_pedido")
    )
    )
    # Regra 7 - Pedidos acima de 300 devem ter frete zero.
    # True = falhou na regra | False = passou na regra.
    .withColumn(
        "dq_regra_07_frete",
        (F.col("valor_total") > 300)
        & (F.col("valor_frete") != 0)
    )   

    # Regra 8 - PENDENTE
    # Depende da tabela de rastreamento para validar se pedidos Cancelados possuem rastreamento ativo.

    # Regra 9 - Pedidos Entregues devem ter atualização de status
    # pelo menos 2 dias após a data do pedido.
    # True = falhou na regra | False = passou na regra.

    .withColumn(
        "dq_regra_09_entrega_prazo",
        F.when(
            F.col("status_pedido") == "Entregue",

            F.col("dt_pedido").isNull()
            | F.col("dt_ultima_atualizacao_status").isNull()
            | (
                F.col("dt_ultima_atualizacao_status")
                < (
                    F.col("dt_pedido")
                    + F.expr("INTERVAL 2 DAYS")
                )
            )
        ).otherwise(False)

    # Regra 10 - PENDENTE
    # Depende da tabela ecommerce_enderecos para validar se
    # id_endereco_entrega existe na tabela de referência.   
)

)

In [0]:
# Confere quantos registros passaram ou falharam na Regra 9.

df_dq.groupBy(
    "dq_regra_09_entrega_prazo"
).count().show()

In [0]:
# Exibe uma amostra dos registros já avaliados pelas regras de Data Quality.

display(
    df_dq.limit(200)
)

**silver_processed_at**

In [0]:
# Registra o momento em que o registro foi processado pela camada Silver.

df_dq = df_dq.withColumn(
    "silver_processed_at",
    F.current_timestamp()
)

In [0]:
# Confere se a coluna silver_processed_at foi adicionada corretamente.

df_dq.select(
    "id_pedido",
    "data_carga",
    "silver_processed_at"
).show(10, truncate=False)

In [0]:
df_dq.printSchema()

In [0]:
# Prepara o DataFrame que será persistido na camada Silver.
# Remove apenas a coluna auxiliar usada para detectar duplicidade.

df_silver = df_dq.drop(
    "qtd_id_pedido"
)

In [0]:
# Confere o schema final antes da persistência na Silver.

df_silver.printSchema()

In [0]:
# Confere uma amostra dos dados preparados para a Silver.

display(
    df_silver.limit(10)
)

In [0]:
# Configurações

silver_path = (
    "abfss://squad1@internshipdatalake.dfs.core.windows.net/"
    "silver/ecommerce_pedidos"
)

In [0]:
# Define o caminho físico da tabela ecommerce_pedidos na camada Silver.

silver_path = (
    "abfss://squad1@internshipdatalake.dfs.core.windows.net/"
    "silver/ecommerce_pedidos"
)

print(silver_path)

In [0]:
# Configura o Spark para acessar o ADLS Gen2 via OAuth.
# As credenciais são recuperadas do Secret Scope e não ficam expostas no código.

storage_account = "internshipdatalake"

client_id = dbutils.secrets.get(
    scope="mercadata-adls-oauth",
    key="client-id"
)

tenant_id = dbutils.secrets.get(
    scope="mercadata-adls-oauth",
    key="tenant-id"
)

client_secret = dbutils.secrets.get(
    scope="mercadata-adls-oauth",
    key="client-secret"
)

spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
    client_id
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
    client_secret
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

In [0]:
dbutils.fs.ls(silver_path)